# 🎟️ Credential Vending

This notebook demonstrates how to setup RBAC and credential vending.
It assumes the infrastructure has already been initialized via `setup.ipynb`.

We will:
1. Define two distinct principals: **Ali** and **Ahmed**.
2. Assign them specific organizational responsibilities using Principal Roles: `data_engineer` and `data_scientist`.
3. Bind RBAC policies mapping back to the existing medallion architecture namespaces (`bronze`, `silver`, `gold`).

**Prerequisites:** Run `setup.ipynb` first.

---
## ⚙️ Step 1 — Authentication & Setup Connection

In [9]:
import requests

# --- Polaris ---
POLARIS_URL = "http://polaris:8181"
POLARIS_CLIENT_ID = "root"
POLARIS_CLIENT_SECRET = "polaris-secret"
CATALOG_NAME = "lakehouse"

def get_polaris_token(client_id=POLARIS_CLIENT_ID, client_secret=POLARIS_CLIENT_SECRET):
    token_resp = requests.post(
        f"{POLARIS_URL}/api/catalog/v1/oauth/tokens",
        data={
            "grant_type": "client_credentials",
            "client_id": client_id,
            "client_secret": client_secret,
            "scope": "PRINCIPAL_ROLE:ALL",
        },
    )
    if token_resp.ok:
        return token_resp.json()["access_token"]
    else:
        raise Exception(f"Failed to get token: {token_resp.text}")

ROOT_TOKEN = get_polaris_token()
headers = {"Authorization": f"Bearer {ROOT_TOKEN}", "Content-Type": "application/json"}
print("✅ Authenticated to Polaris as root")

✅ Authenticated to Polaris as root


---
## 📖 Polaris RBAC Core Concepts

Before creating our users, let's briefly define how Polaris structures its Access Control:
- **Securable Object**: The distinct entity being protected inside Polaris (e.g., a Catalog, a Namespace like `gold`, or a specific Table). 
- **Principal**: An authenticated identity or service account connecting to Polaris. _(e.g., Ali or Ahmed)_
- **Principal Role**: A logical role or job function assigned directly to a principal to group them easily. _(e.g., Data Engineer or Data Scientist)_
- **Catalog Role**: A role scoped to a specific catalog that actively holds the explicit permissions/privileges on securable objects within that catalog.
- **Privilege**: The exact action allowed on a securable object. _(e.g., `TABLE_READ_DATA`, `NAMESPACE_READ_PROPERTIES`)_

RBAC flows like this:
`Principal (Ali) -> Principal Role (data_engineer) -> Catalog Role (catalog_contributor) -> Privileges (TABLE_READ_DATA) -> Securable Object (Namespace: bronze/silver/gold)`

---
## 🔐 Step 2 — Set up RBAC (Principals & Roles)

We define our two users: **Ali**, a Data Engineer, and **Ahmed**, a Data Scientist.

In [10]:
def create_principal(name):
    payload = {"principal": {"name": name}}
    r = requests.post(f"{POLARIS_URL}/api/management/v1/principals", headers=headers, json=payload)
    if r.status_code in (200, 201):
        creds = r.json()["credentials"]
        print(f"✅ Principal '{name}' created. ClientID: {creds['clientId']}")
        return creds
    elif r.status_code == 409:
        print(f"⚠️ Principal '{name}' already exists. Recreating it to get fresh secrets...")
        requests.delete(f"{POLARIS_URL}/api/management/v1/principals/{name}", headers=headers)
        return create_principal(name)
    else:
        print(f"❌ Failed to create principal '{name}': {r.text}")
        return None

ali_creds = create_principal("ali")
ahmed_creds = create_principal("ahmed")

⚠️ Principal 'ali' already exists. Recreating it to get fresh secrets...
✅ Principal 'ali' created. ClientID: 41b02edc19056a7c
⚠️ Principal 'ahmed' already exists. Recreating it to get fresh secrets...
✅ Principal 'ahmed' created. ClientID: 5f236536151e08c5


In [12]:
def create_catalog_role(role_name):
    payload = {"catalogRole": {"name": role_name}}
    r = requests.post(f"{POLARIS_URL}/api/management/v1/catalogs/{CATALOG_NAME}/catalog-roles", headers=headers, json=payload)
    if r.status_code in (200, 201):
        print(f"✅ Catalog role '{role_name}' created.")
    elif r.status_code == 409:
        print(f"⚠️ Catalog role '{role_name}' already exists.")
    else:
        print(f"❌ Failed to create catalog role '{role_name}': {r.text}")

def create_principal_role(role_name):
    payload = {"principalRole": {"name": role_name}}
    r = requests.post(f"{POLARIS_URL}/api/management/v1/principal-roles", headers=headers, json=payload)
    if r.status_code in (200, 201):
        print(f"✅ Principal role '{role_name}' created.")
    elif r.status_code == 409:
        print(f"⚠️ Principal role '{role_name}' already exists.")
    else:
        print(f"❌ Failed to create principal role '{role_name}': {r.text}")

create_catalog_role("catalog_contributor")
create_catalog_role("catalog_reader")

create_principal_role("data_engineer")
create_principal_role("data_scientist")

⚠️ Catalog role 'catalog_contributor' already exists.
⚠️ Catalog role 'catalog_reader' already exists.
⚠️ Principal role 'data_engineer' already exists.
⚠️ Principal role 'data_scientist' already exists.


Here, we explicitly map permissions. **Ali** (Data Engineer) gets access to `bronze`, `silver`, and `gold`. **Ahmed** (Data Scientist) only gets access to the highly curated `gold` models.

In [14]:
def grant_privilege_to_catalog_role(namespace, catalog_role, privileges=["NAMESPACE_READ_PROPERTIES", "TABLE_LIST", "TABLE_READ_DATA"]):
    for priv in privileges:
        url = f"{POLARIS_URL}/api/management/v1/catalogs/{CATALOG_NAME}/catalog-roles/{catalog_role}/grants"
        payload = {"grant": {"type": "namespace", "namespace": [namespace], "privilege": priv}}
        r = requests.put(url, headers=headers, json=payload)
        if r.status_code in (200, 201, 204):
             print(f"✅ Granted {priv} on namespace '{namespace}' to '{catalog_role}'")
        else:
             print(f"❌ Failed to grant {priv} on '{namespace}' to '{catalog_role}': {r.text}")

def bind_roles(catalog_role, principal_role, principal_name):
    r1 = requests.put(f"{POLARIS_URL}/api/management/v1/principal-roles/{principal_role}/catalog-roles/{CATALOG_NAME}", headers=headers, json={"name": catalog_role})
    r2 = requests.put(f"{POLARIS_URL}/api/management/v1/principals/{principal_name}/principal-roles", headers=headers, json={"name": principal_role})
    if r1.ok and r2.ok:
        print(f"✅ Successfully bound Path: [CatalogRole: '{catalog_role}'] -> [PrincipalRole: '{principal_role}'] -> [User: '{principal_name}']")
    else:
        print("❌ Failed to bind roles")

# Grants for Ali (Data Engineer across All Namespaces)
print("GRANTING DATA ENGINEER (ALI) LEASES:")
grant_privilege_to_catalog_role("bronze", "catalog_contributor")
grant_privilege_to_catalog_role("silver", "catalog_contributor")
grant_privilege_to_catalog_role("gold", "catalog_contributor")
bind_roles("catalog_contributor", "data_engineer", "ali")

# Grants for Ahmed (Data Scientist on Gold Only)
print("\nGRANTING DATA SCIENTIST (AHMED) LEASES:")
grant_privilege_to_catalog_role("gold", "catalog_reader")
bind_roles("catalog_reader", "data_scientist", "ahmed")

GRANTING DATA ENGINEER (ALI) LEASES:
✅ Granted NAMESPACE_READ_PROPERTIES on namespace 'bronze' to 'catalog_contributor'
✅ Granted TABLE_LIST on namespace 'bronze' to 'catalog_contributor'
✅ Granted TABLE_READ_DATA on namespace 'bronze' to 'catalog_contributor'
✅ Granted NAMESPACE_READ_PROPERTIES on namespace 'silver' to 'catalog_contributor'
✅ Granted TABLE_LIST on namespace 'silver' to 'catalog_contributor'
✅ Granted TABLE_READ_DATA on namespace 'silver' to 'catalog_contributor'
✅ Granted NAMESPACE_READ_PROPERTIES on namespace 'gold' to 'catalog_contributor'
✅ Granted TABLE_LIST on namespace 'gold' to 'catalog_contributor'
✅ Granted TABLE_READ_DATA on namespace 'gold' to 'catalog_contributor'
✅ Successfully bound Path: [CatalogRole: 'catalog_contributor'] -> [PrincipalRole: 'data_engineer'] -> [User: 'ali']

GRANTING DATA SCIENTIST (AHMED) LEASES:
✅ Granted NAMESPACE_READ_PROPERTIES on namespace 'gold' to 'catalog_reader'
✅ Granted TABLE_LIST on namespace 'gold' to 'catalog_reader'
✅ 

---
## 🧪 The RBAC Experiment (Metadata Isolation)

**Objective**: Demonstrate that lacking appropriate catalog roles prevents users from discovering basic metadata, such as namespaces or table listings.

**Action**: Attempt to list tables within the `bronze` raw zone utilizing **Ahmed's** (Data Scientist) restricted credentials.

**Expected Outcome**: Polaris actively rejects the request with a "Forbidden" or "Not Found" error. Unauthorized users are halted at the catalog level before any physical S3 storage interactions occur.

In [15]:
def verify_metadata_rbac(principal_creds, namespace):
    token = get_polaris_token(principal_creds["clientId"], principal_creds["clientSecret"])
    auth_headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
    
    url = f"{POLARIS_URL}/api/catalog/v1/{CATALOG_NAME}/namespaces/{namespace}/tables"
    r = requests.get(url, headers=auth_headers)
    
    if r.status_code == 200:
        tables = r.json().get("identifiers", [])
        print(f"✅ Metadata Access allowed. Tables in '{namespace}': {[(t['namespace'][0], t['name']) for t in tables]}")
    elif r.status_code in (403, 404):
        print(f"🚫 Metadata Isolation triggered! Polaris denied discovery access (Status {r.status_code}): {r.text}")
    else:
        print(f"❌ Unexpected response: {r.status_code} - {r.text}")


print("🧑‍🔬 Ahmed (Data Scientist) trying to discover 'bronze' raw orders:")
verify_metadata_rbac(ahmed_creds, "bronze")

print("\n🧑‍🔬 Ahmed (Data Scientist) trying to discover 'gold' summary tables:")
verify_metadata_rbac(ahmed_creds, "gold")

🧑‍🔬 Ahmed (Data Scientist) trying to discover 'bronze' raw orders:
🚫 Metadata Isolation triggered! Polaris denied discovery access (Status 403): {"error":{"message":"Principal 'ahmed' with activated PrincipalRoles '[data_scientist]' and activated grants via '[data_scientist, catalog_reader]' is not authorized for op LIST_TABLES","type":"ForbiddenException","code":403}}

🧑‍🔬 Ahmed (Data Scientist) trying to discover 'gold' summary tables:
✅ Metadata Access allowed. Tables in 'gold': []


---
## 🧪 Credential Vending Experiment (Data Exfiltration Prevention)

**What is Credential Vending?**
Instead of giving compute engines (like Trino or Spark) permanent, long-lived AWS IAM keys to access your entire S3 bucket, **Credential Vending** dynamically generates temporary, scoped access tokens. When an engine requests a table, Polaris verifies the user's RBAC permissions and vends back a short-lived STS (Security Token Service) token that *only* grants access to the exact S3 prefix where that specific table's data resides.

**Objective**: Objective the engine can't physically touch the data files in Object Storage (S3) unless they pass through Polaris to receive a temporary, catalog-issued lease.

**Action**: Attempt to query `gold.daily_sales_summary` as **Ahmed**, and `silver.cleansed_orders` as **Ali**.

**Internal Trace**: Show how Polaris calculates exactly what S3 prefix a user needs access to and injects a temporary scoped token string for SeaweedFS directly into the Iceberg REST response's configuration block.

**Expected Outcome**: The REST client logs the dynamically generated `s3.access-key-id` config proving that AWS credentials are vended specifically for this discrete table request. This guarantees Trino only touches data when a Polaris lease is actively attached.

In [16]:
def verify_vended_credentials(principal_creds, namespace, table):
    # 1. Get token for the principal
    token = get_polaris_token(principal_creds["clientId"], principal_creds["clientSecret"])
    auth_headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
    
    # 2. Call Load Table on Iceberg REST API
    url = f"{POLARIS_URL}/api/catalog/v1/{CATALOG_NAME}/namespaces/{namespace}/tables/{table}"
    r = requests.get(url, headers=auth_headers)
    
    if r.status_code == 200:
        data = r.json()
        print(f"✅ Successfully retrieved physical read metadata for table '{namespace}.{table}'")
        
        # The magic of credential vending is in `config` dynamically returned by the Load Table Payload
        if "config" in data:
            config = data["config"]
            print("\n--- 🎫 Vended Storage Credentials Acquired! ---")
            print(f"🔑 Access Key: {config.get('s3.access-key-id', 'Not Provided')}")
            secret = config.get('s3.secret-access-key')
            if secret:
                print(f"🤫 Secret Key: {secret[:4]}...{secret[-4:]}")
            print(f"🎯 Authorized Data Path Scope: {data['tableMetadata'].get('location')}")
            session_token = config.get('s3.session-token')
            if session_token:
                print(f"📜 Session AWS Token: {session_token[:10]}...")
            print("----------------------------------------------\n")
        else:
            print("⚠️ No config block returned. Vended credentials might not be enabled on the server/catalog side.")
    
    elif r.status_code in (403, 404):
        print(f"🚫 Access Denied: Cannot access '{namespace}.{table}'. Status: {r.status_code}")
    else:
        print(f"❌ Error: {r.status_code} - {r.text}")

print("🧑‍🔬 Ahmed (Data Scientist) requests physical access to gold.daily_sales_summary:\n")
verify_vended_credentials(ahmed_creds, "gold", "daily_sales_summary")

print("\n🧑‍🔬 Ahmed (Data Scientist) illegally requests physical access to silver.cleansed_orders:\n")
verify_vended_credentials(ahmed_creds, "silver", "cleansed_orders")

print("\n🧑‍🔧 Ali (Data Engineer) requests physical access to silver.cleansed_orders:\n")
verify_vended_credentials(ali_creds, "silver", "cleansed_orders")

🧑‍🔬 Ahmed (Data Scientist) requests physical access to gold.daily_sales_summary:

🚫 Access Denied: Cannot access 'gold.daily_sales_summary'. Status: 404

🧑‍🔬 Ahmed (Data Scientist) illegally requests physical access to silver.cleansed_orders:

🚫 Access Denied: Cannot access 'silver.cleansed_orders'. Status: 404

🧑‍🔧 Ali (Data Engineer) requests physical access to silver.cleansed_orders:

🚫 Access Denied: Cannot access 'silver.cleansed_orders'. Status: 404
